# UK Contracts Finder — Scrape & Upload to Notion

**Where this comes from:** UK Contracts Finder - the UK government's register of public sector contract opportunities. This notebook reads from `output_data.json`, which a separate fetch script produces before this notebook runs.

**What this notebook does:** takes the contracts already filtered into `output_data.json`, re-scrapes the live search results page for full display fields (value, location, CPV codes, description), and adds new ones to the unified Notion database.

**Filters applied:**
- CPV codes: same consultancy/research list used across all sources (applied upstream, in the fetch step that produces `output_data.json`)
- Blocked keywords: notices are hard-excluded if the title or description matches any term in `blocked_words.py` (shared blocklist, sources/ root) - see that file for the current list and notes on match behaviour


In [1]:
import requests 
from datetime import datetime, timezone
import json
import requests
import pandas as pd
from datetime import date
from parsel import Selector
import os
import csv

In [2]:
import os

NOTION_TOKEN = os.environ["NOTION_TOKEN"]


DATABASE_ID= '334701e728cb8096a94cebc0985684a2'

headers = {
    "Authorization": "Bearer " + NOTION_TOKEN,
    "Content-Type": "application/json",
    "Notion-Version": "2022-06-28",
}

In [3]:
# Shared keyword blocklist (sources/ root) - added 2026-08-09 per Javiera's feedback
import sys
from pathlib import Path

BLOCKED_WORDS_PATH = Path("../blocked_words.py")
if not BLOCKED_WORDS_PATH.exists():
    raise FileNotFoundError(f"Could not find {BLOCKED_WORDS_PATH.resolve()}")

sys.path.insert(0, str(BLOCKED_WORDS_PATH.parent.resolve()))
from blocked_words import is_blocked, blocked_keyword_hits


In [4]:
def get_pages(num_pages=None):
    """
    If num_pages is None, get all pages, otherwise just the defined number.
    """
    url = f"https://api.notion.com/v1/databases/{DATABASE_ID}/query"

    get_all = num_pages is None
    page_size = 100 if get_all else num_pages

    payload = {"page_size": page_size}
    response = requests.post(url, json=payload, headers=headers)

    data = response.json()


    results = data["results"]
    while data["has_more"] and get_all:
        payload = {"page_size": page_size, "start_cursor": data["next_cursor"]}
        url = f"https://api.notion.com/v1/databases/{DATABASE_ID}/query"
        response = requests.post(url, json=payload, headers=headers)
        data = response.json()
        results.extend(data["results"])

    return results

In [5]:
def create_page(data: dict) -> bool:
    try:
        create_url = "https://api.notion.com/v1/pages"

        # Replace None values with an empty string
        data = {key: value if value is not None else "" for key, value in data.items()}

        payload = {"parent": {"database_id": DATABASE_ID}, "properties": data}

        res = requests.post(create_url, headers=headers, json=payload)
        if not res.ok:
            print(f"❌ Notion error: {res.status_code} {res.text[:500]}")
            return False
        return True
    except Exception as e:
        print(f"Error creating page: {e}")
        return False


In [6]:
with open('output_data.json', 'r') as f:
    data = json.load(f)

filtered_titles_set = {entry['item']['title'] for entry in data['noticeList']}

The code below is only ran when restarting process. This creates the initial CSV used for checking whether contract is available by fetching all the contract titles already on notion. Once this has been ran for the first time, the csv will be created and it is no long needed. 

In [7]:

# def get_database_entries():
#     """Fetch all pages from a Notion database with pagination."""
#     url = f"https://api.notion.com/v1/databases/{DATABASE_ID}/query"
#     all_entries = []
#     payload = {}

#     while True:
#         response = requests.post(url, headers=headers, json=payload)
        
#         if response.status_code != 200:
#             print(f"Error fetching database: {response.text}")
#             return []

#         data = response.json()
#         all_entries.extend(data.get("results", []))
        
#         # Check if there's more data
#         next_cursor = data.get("next_cursor")
#         if not next_cursor:
#             break  # No more pages
        
#         # Update payload for next request
#         payload = {"start_cursor": next_cursor}

#     return all_entries


# database_entries = get_database_entries()


# contract_titles = [
#     entry['properties']['Name']['title'][0]['text']['content']
#     for entry in database_entries
#     if 'Name' in entry['properties'] and entry['properties']['Name']['title']]


# pd.DataFrame(contract_titles, columns=['Title']).to_csv('contract_titles.csv', index=False)

# print(f"Total contracts saved: {len(contract_titles)}")


In [8]:
# creates a set of titles that have already been scraped 

contract_titles = pd.read_csv('contract_titles.csv')
contract_titles = set(contract_titles['Title'])

len(contract_titles)


852

In [9]:
def extract_cpv_codes(contract_link):
    """Scrape CPV codes from the given contract page."""
    try:
        response = requests.get(contract_link)
        if response.status_code != 200:
            print(f"Failed to fetch {contract_link}: {response.status_code}")
            return []

        sel = Selector(text=response.text)
        cpv_codes = sel.xpath('//ul/li/p/text()').re(r'\d{7,8}')  # Extract CPV codes
        return cpv_codes
    except Exception as e:
        print(f"Error scraping {contract_link}: {e}")
        return []

In [10]:
from datetime import datetime, timezone
import requests
from parsel import Selector
import pandas as pd
import os

contracts_notion_df = pd.DataFrame(columns=["Contract Name"])

base_url = "https://www.contractsfinder.service.gov.uk/Search/Results?page="

# Store all contracts in a list
all_contracts = []

# function to ensure all data type formats are accounted for

def parse_closing_date(closing_date):
    """Parses the closing date, returning None if no valid date is found."""
    if not closing_date:
        return None
    
    try:
        return datetime.strptime(closing_date, "%d %B %Y, %I:%M%p").astimezone(timezone.utc).isoformat()
    except ValueError:
        try:
            return datetime.strptime(closing_date, "%d %B %Y, %I%p").astimezone(timezone.utc).isoformat()
        except ValueError:
            try:
                return datetime.strptime(closing_date, "%d %B %Y").astimezone(timezone.utc).isoformat()
            except ValueError:
                return None


for page_number in range(1, 50):  # Loop through pages
    url = base_url + str(page_number)
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Failed on {page_number}, status code: {response.status_code}")
        continue  

    sel = Selector(text=response.text)
    contract_nodes = sel.xpath('//div[@class="search-result"]')
    
    for node in contract_nodes:
        name = node.xpath('.//div[@class="search-result-header"]/@title').get()
        
        if name not in filtered_titles_set:
            continue

        if name in contract_titles:
            continue
        
        # Extract other attributes
        procurement_stage = node.xpath('.//div[@class="search-result-entry"]/text()').get()
        contract_value = node.xpath('.//strong[contains(text(), "Contract value")]/following-sibling::text()').get()
        contract_location = node.xpath('.//strong[contains(text(), "Contract location")]/following-sibling::text()').get()
        client = node.xpath('.//div[@class="search-result-sub-header wrap-text"]/text()').get()
        closing_date = node.xpath('.//strong[contains(text(), "Closing")]/following-sibling::text()').get()
        contract_link = node.xpath('.//div[@class="search-result-header"]//a/@href').get()
        description = node.xpath('.//div[@class="wrap-text"]/span[@title]/@title | .//div[@class="wrap-text"]/text()').get()

        # Cleaning
        procurement_stage = procurement_stage.strip() if procurement_stage else None
        contract_value = contract_value.strip() if contract_value else 'Not Disclosed'
        contract_location = contract_location.strip() if contract_location else None
        client = client.strip() if client else 'Not Disclosed'
        closing_date = closing_date.strip() if closing_date else None
        contract_link = contract_link.strip() if contract_link else None
        if is_blocked(name, description):
            hits = blocked_keyword_hits(name, description)
            print(f"⛔ Skipping blocked keyword ({', '.join(hits)}): {name}")
            continue

        description = description.strip() if description else 'Not Disclosed'
        date_added = datetime.now().astimezone(timezone.utc).isoformat()
        cpv_codes = extract_cpv_codes(contract_link)
        cpv_codes_text = ", ".join(cpv_codes) if cpv_codes else "Not Available"

        # Parse closing date
        closing_date_iso = parse_closing_date(closing_date)
        
        # Prepare data
        data = {
            "Name": {"title": [{"text": {"content": name}}]},
            "Value": {"rich_text": [{"text": {"content": contract_value}}]},
            "Client": {"rich_text": [{"text": {"content": client}}]},
            "Location": {
                "rich_text": [
                    {"text": {"content": contract_location or "Not Disclosed"}}
                ]
            },
            "Procurement Stage": {"select": {"name": procurement_stage}},
            "Closing Date": {"date": {"start": closing_date_iso, "end": None}} if closing_date_iso else {"date": None},
            "Description": {"rich_text": [{"text": {"content": description}}]},
            "Contract Link": {"url": contract_link},
            "Date Added": {"date": {"start": date_added, "end": None}},
            "Review Status": {"select": {"name": "Not Reviewed"}},
            "Contract Status": {"select": {"name": "Open"}},
            "Reviewed By": {"select": {"name": "N/A"}},
            "CPV Codes": {"rich_text": [{"text": {"content": cpv_codes_text}}]},
            "Source": {"select": {"name": "UK Contracts Finder"}}

        }


        try:
            all_contracts.append(data)
        except Exception as e:
            print(f"Error preparing contract '{name}': {e}")

# Reverse order before uploading to Notion
all_contracts.reverse()

# Upload, then record ONLY successful uploads - and only THEN save the CSV.
# (Previously the CSV was written before any Notion API calls were even made,
# so a failed create_page() call still got marked as uploaded and lost to dedup.)
uploaded_rows = []
for contract in all_contracts:
    contract_name = contract["Name"]["title"][0]["text"]["content"]
    success = create_page(contract)
    if success:
        uploaded_rows.append({'Contract Name': contract_name})
    else:
        print(f"⚠️ Failed to upload, not recording in dedup CSV: {contract_name}")

if uploaded_rows:
    contracts_notion_df = pd.concat([contracts_notion_df, pd.DataFrame(uploaded_rows)], ignore_index=True)
    contracts_notion_df.to_csv('contract_titles.csv', mode='a', header=not os.path.exists('contract_titles.csv'), index=False)

print(f'DONE - uploaded {len(uploaded_rows)} of {len(all_contracts)} contracts')

Failed to fetch https://www.contractsfinder.service.gov.uk/notice/6a7c818a-2df4-4b60-b934-4c48a9f9ea3f?origin=SearchResults&p=1: 403


Failed to fetch https://www.contractsfinder.service.gov.uk/notice/c684c2d3-71be-4a22-bea1-56816b71f563?origin=SearchResults&p=1: 403


Failed on 18, status code: 429


Failed on 19, status code: 429


Failed on 20, status code: 429


...

Failed on 46, status code: 429


Failed on 47, status code: 429


Failed on 48, status code: 429


Failed on 49, status code: 429


DONE - uploaded 2 of 2 contracts


The below code is a quick fix is something goes wrong in the scraping process and there are many duplicates. It can be ran to delete duplicates, and it keeps the first instance of the duplicate title.

In [11]:
# REMOVE DUPLICATES 

# import requests

# def get_database_entries():
#     """Fetch all pages from a Notion database."""
#     url = f"https://api.notion.com/v1/databases/{DATABASE_ID}/query"
#     response = requests.post(url, headers=headers)
#     if response.status_code == 200:
#         return response.json().get("results", [])
#     else:
#         print(f"Error fetching database: {response.text}")
#         return []

# def delete_page(page_id):
#     """Deletes a page from Notion by setting 'archived' to True."""
#     url = f"https://api.notion.com/v1/pages/{page_id}"
#     data = {"archived": True}
#     response = requests.patch(url, headers=headers, json=data)
#     if response.status_code == 200:
#         print(f"Deleted page {page_id}")
#     else:
#         print(f"Error deleting page {page_id}: {response.text}")

# def remove_duplicate_contracts():
#     """Identifies duplicate contract titles and deletes all but the first instance."""
#     entries = get_database_entries()
#     seen_titles = {}
    
#     for entry in entries:
#         title_property = entry["properties"].get("Name", {})
#         if title_property and title_property["type"] == "title":
#             title_text = title_property["title"][0]["plain_text"] if title_property["title"] else ""
#             page_id = entry["id"]

#             if title_text in seen_titles:
#                 # If title is already seen, delete this duplicate
#                 delete_page(page_id)
#             else:
#                 # Store the first occurrence of this title
#                 seen_titles[title_text] = page_id

# # Run the script
# remove_duplicate_contracts()
